**Notebook: Building an AI Agent (Module 7)**
- OCBC AI Creator Masterclass, Day 2

___

This notebook is the hands-on part of Module 7, in two parts: **7.1–7.4 before lunch** (about 40 minutes) and
**7.5–7.7 after lunch** (about 28 minutes). After each part comes part of the exercise `day2/exercises/m7_exercise.py`.
One agent question uses several model calls: run each cell **once**.

# Background

In Module 6 you ran the round trip by hand, once: the LLM asked for one tool, your code ran it, the LLM answered.
Real RM questions need more than one step: *"Which bank has the best ROE, and how did it trade last week?"* needs the
ranking **first**, and only then the prices of the winner. An **agent** repeats **plan → act → observe** until it can
answer, and Pydantic AI's `Agent` runs that loop for us:

- **7.1 The agent.** Model, tools and rules in one `Agent`.
- **7.2 Inside the loop.** Reading every step in `result.all_messages()`.
- **7.3 The rules.** The system prompt, and checking answers against tool results.
- **7.4 Structured answers.** `output_type`.
- **7.5 Ready-made tools.** Web search, Gmail and image generation, as in `learning-agent/`.
- **7.6 Approval before actions.** Pause, review, approve, edit or reject.
- **7.7 Memory and limits.** Message history, and what keeps an agent safe.

| Step on the ladder | Who decides WHICH data | Who decides WHEN |
|---|---|---|
| Tool calling (M6) | The LLM, one choice | The user asks, the LLM picks |
| **Agent (M7)** | **The LLM, step by step** | **The LLM** |
| Scheduled agent (M9) | The LLM | The scheduler or a condition |

## Training Objectives

- Build an agent with Pydantic AI's `Agent` from a model, tools and a system prompt
- Read the plan, act and observe steps of a run, and check an answer against the tool results
- Write system-prompt rules and see them change the agent's behaviour
- Get a structured answer with `output_type`
- Add ready-made tools (web search, and LangChain's Gmail toolkit through Pydantic AI's adapter) and wrap another
  service's API (image generation) as a tool
- Pause an action tool for review with `requires_approval`, and approve, edit or reject it
- Use the message history as short-term memory, and explain the limits of an agent

## Checking your answers

**Your turn** cells say what the right result is. Run the next cell once for the **Knowledge Check** questions.

## Before 7.5 (after lunch)

7.5 and 7.6 use three outside services. Each one needs something in the project folder; without it, the guided cells
still run and only that tool is skipped.

| Tool | Needs | Where from |
|---|---|---|
| Web search (DuckDuckGo) | nothing | – |
| Gmail (draft and send) | `credentials.json`, then a one-time Google sign-in that saves `token.json` | your trainer |
| Image generation (Magic Hour) | `MAGIC_API_KEY` in `.env` | your trainer |

Also put your own e-mail address in `.env` as `MY_EMAIL`: the agent sends its test e-mail there.
`credentials.json` and `token.json` are keys, like `.env`: never share them.

In [ ]:
import hashlib

ANSWERS = {'m7_runs': '3784e709a6', 'm7_plan': '4e0a284776', 'm7_check': '5d14d2fdd7', 'm7_pick': '6e4070149a', 'm7_send': '56936f1af4', 'm7_interrupt': '05d7b15cd0', 'm7_restart': '18708c6637', 'm7_when_what': '82227396fd'}

def check_answer(question, answer):
    key = hashlib.sha256(f"{question}:{answer.strip().lower()}".encode()).hexdigest()[:10]
    if answer.strip() == "":
        print("Type a, b, c or d between the quotes, then run the cell again.")
    elif key == ANSWERS[question]:
        print("Correct.")
    else:
        print("Not quite. Read the question again and try another answer.")

In [ ]:
import os
from dotenv import load_dotenv

# Work from the project root (the folder with data/), so file paths are the same as in the .py scripts
while not os.path.exists("data/sample"):
    os.chdir("..")

load_dotenv()  # reads the keys from .env
print("Working folder:", os.getcwd())

# 7.1 The Agent

> **The request**
>
> An RM: **"Which bank has the highest return on equity, and what was its last close?"** Two steps, and the
> second depends on the first.

> **In this section**
>
> - **Key question:** What does an `Agent` need, and what does it do for us?
> - **How we run it:** **Live, about 10 minutes.**
> - **What you will do:** define the Module 6 tools plus one action tool, build the agent, and ask the two-step question.
> - **What carries forward:** `tools`, `SYSTEM_PROMPT` and `agent`, used in every section below.

The same Sectors tools as Module 6, plus `save_note`, an **action** tool: it writes a file, so in 7.6 it will need
approval. `Agent` takes the **model**, the **tools** and the **system prompt** (`instructions=`), and `agent.run(...)`
runs the loop of Module 6 as many rounds as the question needs.

In [ ]:
import requests
from datetime import date, timedelta
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIResponsesModel
from pydantic_ai.providers.openai import OpenAIProvider

BASE_URL = "https://api.sectors.app/v2"
HEADERS = {"Authorization": os.getenv("SECTORS_API_KEY")}
model = OpenAIResponsesModel(
    os.getenv("OPENAI_MODEL", "gpt-6.1-sol"),
    provider=OpenAIProvider(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("OPENAI_URL")),
)


def get_company_overview(ticker: str) -> dict:
    """Get the profile and latest closing price of one company on the Indonesia Stock Exchange.
    Use it for questions about a single company, e.g. "How did NISP close?".

    Args:
        ticker: 4-letter IDX code, e.g. BBCA, BBRI, BMRI, NISP.
    """
    response = requests.get(f"{BASE_URL}/company/report/{ticker}/", headers=HEADERS,
                            params={"sections": "overview"}, timeout=30)
    return response.json()


def get_daily_prices(ticker: str, days: int = 10) -> list:
    """Get the daily closing prices and volumes of one IDX company for the last `days` calendar days.
    Use it for questions about how a stock moved recently, e.g. "How did BMRI do this week?".

    Args:
        ticker: 4-letter IDX code, e.g. BMRI.
        days: how many calendar days back to look (a week is 7).
    """
    end = date.today()
    response = requests.get(f"{BASE_URL}/daily/{ticker}/", headers=HEADERS,
                            params={"start": str(end - timedelta(days=days)), "end": str(end)}, timeout=30)
    return response.json()


def get_top_banks(metric: str = "market_cap", n: int = 5) -> dict:
    """Rank Indonesian listed banks by a metric, highest first.
    Use it for ranking questions such as "Which banks are the largest?" or "Which bank has the highest ROE?".

    Args:
        metric: one of market_cap, roe_ttm (return on equity), pe_ttm (price/earnings), pb_mrq (price/book), yield_ttm (dividend yield).
        n: how many banks to return.
    """
    params = {"where": "sub_sector='banks'", "order_by": f"-{metric}", "limit": n, "include_query_values": "true"}
    response = requests.get(f"{BASE_URL}/companies/", headers=HEADERS, params=params, timeout=30)
    return response.json()


def save_note(filename: str, text: str) -> str:
    """Save a short note to a text file in the output folder. Use it only when the user asks to save something."""
    os.makedirs("output/notes", exist_ok=True)
    with open(f"output/notes/{filename}", "w", encoding="utf-8") as f:
        f.write(text)
    return f"Note saved to output/notes/{filename}"


tools = [get_company_overview, get_daily_prices, get_top_banks, save_note]

SYSTEM_PROMPT = """You are the IDX Market Assistant for relationship managers at a bank in Indonesia.
Use the tools to get data; never guess a number. Say the date of every price you quote.
Never give buy, sell or hold advice. Only save a note when the user asks."""

agent = Agent(model, tools=tools, instructions=SYSTEM_PROMPT)

result = await agent.run("Which bank has the highest return on equity, and what was its last close?")
print(result.output)

#### Knowledge Check: Who runs the tools?

In that run, who sent the requests to Sectors?

- **a.** The model, from the provider's servers
- **b.** Our Python program: the agent runs the tools the LLM asks for
- **c.** Sectors, after reading the prompt
- **d.** Nobody: the LLM knew the numbers

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m7_runs", answer)

# 7.2 Inside the Loop

> **The request**
>
> Your manager: **"Before an RM sees an answer, I want to see how the agent got there."**

> **In this section**
>
> - **Key question:** How do we read each plan, act and observe step of a run?
> - **How we run it:** **Live, about 12 minutes.**
> - **What you will do:** print every part of the run's messages as a step, then run and read a question that needs a chain of steps.
> - **What carries forward:** `show_steps`, used to check every run below.

`result.all_messages()` is the whole run: `ModelRequest`s sent to the model and `ModelResponse`s from it, each made
of parts.

| Part | Step |
|---|---|
| `ToolCallPart` in a `ModelResponse` | **plan**: the LLM decides the next tool and its arguments |
| `ToolReturnPart` in a `ModelRequest` | **act + observe**: our code ran the tool, and this is the result the LLM reads |
| `TextPart` in the last `ModelResponse` | the final answer |

In [ ]:
from pydantic_ai.messages import UserPromptPart, ToolCallPart, ToolReturnPart, TextPart


def show_steps(result):
    for message in result.all_messages():
        for part in message.parts:
            if isinstance(part, UserPromptPart):
                print("USER    ->", str(part.content)[:90])
            elif isinstance(part, ToolCallPart):
                print("PLAN    ->", (part.tool_name, part.args_as_dict()))
            elif isinstance(part, ToolReturnPart):
                print("OBSERVE ->", part.tool_name, ":", str(part.content)[:90], "...")
            elif isinstance(part, TextPart):
                print("ANSWER  ->", part.content[:90])


show_steps(result)

#### Your turn: A chain of steps

Ask the agent *"Which bank has the highest dividend yield, and how did its share price move over the last 7 days?"*
and show the steps. Then answer for yourself: which argument of the second tool call came from the **result** of the
first? The right result is a `get_top_banks` call with `yield_ttm`, then `get_daily_prices` with the ticker that
`get_top_banks` returned and `days` 7: the agent planned the second step only after it observed the first.

In [ ]:
## Your code below



#### Knowledge Check: Plan, act, observe

In the dividend question, why could the agent not ask for both tools in its first step?

- **a.** An `Agent` allows one tool per run
- **b.** The ticker for the price tool was only known after the ranking result came back
- **c.** The price tool needs approval
- **d.** The model can only call one tool per day

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m7_plan", answer)

# 7.3 The Rules: the System Prompt

> **The request**
>
> Your manager: **"Compliance wants every answer to end with where the data came from and its date."**

> **In this section**
>
> - **Key question:** How do the rules change the agent, and how do we check an answer?
> - **How we run it:** **Live, about 8 minutes, then your own rule.**
> - **What you will do:** add a rule to the system prompt, rebuild the agent, and check the numbers in the answer against the tool results.
> - **What carries forward:** `ask`, the helper used below.

The system prompt is where the rules live: what to use, what never to do, how to answer. A rule in the prompt is a
**request** to the model, not a guarantee, so we check: every number in the answer should appear in a `ToolReturnPart`.

In [ ]:
async def ask(agent, question):
    result = await agent.run(question)
    used = [p.tool_name for m in result.all_messages() for p in m.parts if isinstance(p, ToolCallPart)]
    print("Tools used:", used)
    print(result.output)
    return result

#### Your turn: A compliance rule

Write `COMPLIANCE_PROMPT`: the `SYSTEM_PROMPT` plus a rule that every answer ends with a line
`Source: Sectors, data as of <date>` and that numbers are given in IDR with thousands separators. Build `compliance_agent`
with it, ask *"How did NISP close on its last trading day?"*, and print the close found in the `ToolReturnPart` next to the
answer. The right result ends with `Source: Sectors, data as of 2026-…`, and the close in the answer is the
`last_close_price` in the tool result.

In [ ]:
## Your code below



#### Knowledge Check: Checking a number

An answer says *"BBCA closed at IDR 5,100"*. How do you check it?

- **a.** Ask the agent again
- **b.** Look for the number in the run's `ToolReturnPart`s; if it is not there, the model made it up
- **c.** Trust it: the system prompt forbids guessing
- **d.** Compare it with yesterday's digest

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m7_check", answer)

# 7.4 Structured Answers

> **The request**
>
> Your manager: **"Module 9 will pass the agent's answer to code. Give it fields, like in Module 4."**

> **In this section**
>
> - **Key question:** How do we get an agent's answer as an object?
> - **How we run it:** **Live, about 6 minutes, then your own schema.**
> - **What you will do:** pass a Pydantic model as `output_type` and read `result.output`.
> - **What carries forward:** the `output_type` pattern used in Module 9.

In [ ]:
from pydantic import BaseModel, Field

class BankAnswer(BaseModel):
    ticker: str
    last_close: int
    close_date: str
    summary: str = Field(description="One factual sentence, no advice")

structured_agent = Agent(model, tools=tools, instructions=SYSTEM_PROMPT, output_type=BankAnswer)
structured = await structured_agent.run("How did NISP close on its last trading day?")
structured.output

#### Your turn: A ranking schema

Design `RankedBank` (`ticker`, `value` as float) and `RankingAnswer` (`metric`, `banks` as a list of `RankedBank`).
Build an agent with `output_type=RankingAnswer` and ask *"Which three banks have the highest dividend yield?"* Print each
bank and value. The right result is `metric` about dividend yield and 3 banks with their yields, the same banks as in
the `get_top_banks` result.

In [ ]:
## Your code below



**End of part 1.** Next: exercise part 1 (`day2/exercises/m7_exercise.py`, TODO 1). After lunch: 7.5 to 7.7.

# 7.5 Ready-made Tools

> **The request**
>
> Your manager: **"The RMs also want the news from the web, a picture for the top of the digest, and the digest in
> their inbox. Do we write all of that ourselves?"**

> **In this section**
>
> - **Key question:** Which tools can we take ready-made, and how do we add a service that has no ready-made tool?
> - **How we run it:** **Live, about 8 minutes, then your own illustration.**
> - **What you will do:** load a web search tool and the Gmail toolkit, wrap an image API in a plain function, and give them all to one agent.
> - **What carries forward:** `web_search`, `generate_image`, `gmail` and `DESK_PROMPT`, used in 7.6.

Pydantic AI ships a few **ready-made tools** for common services, such as DuckDuckGo web search. It has no Gmail tools
of its own, but LangChain's `langchain_google_community` has a **toolkit** for Gmail (several tools from one sign-in),
and Pydantic AI's LangChain adapter lets an agent use any ready-made LangChain tool. A ready-made tool is the same
thing as ours: a name, a description the LLM reads, and arguments. Print them before you trust them.

In [ ]:
from pydantic_ai.common_tools.duckduckgo import duckduckgo_search_tool

web_search = duckduckgo_search_tool()
print(web_search.name, "|", web_search.description)

For Gmail, `GmailToolkit()` signs in with `credentials.json` (the first run opens a browser and saves `token.json`) and
returns five tools. The agent needs only two of them: a draft, and sending. Fewer tools means fewer wrong choices, and
no access to mail it does not need, the same idea as `SELECTED` in Module 8. `LangChainToolset` turns the two
LangChain tools into tools our agent can use.

In [ ]:
from langchain_google_community import GmailToolkit
from pydantic_ai.ext.langchain import LangChainToolset

if os.path.exists("credentials.json"):
    all_gmail = GmailToolkit().get_tools()
    print("The toolkit has:", [t.name for t in all_gmail])
    gmail_tools = [t for t in all_gmail if t.name in ("create_gmail_draft", "send_gmail_message")]
else:
    gmail_tools = []
    print("No credentials.json in the project folder: the Gmail tools are skipped.")
print("We keep:", [t.name for t in gmail_tools])

gmail = LangChainToolset(gmail_tools)   # the LangChain tools, ready for our agent

When a service has a Python package but no ready-made tool, wrap it in a plain function yourself, exactly like the
Sectors tools.
This is the image tool from `learning-agent/`, which calls Magic Hour, saves the picture, and tells the agent where:

In [ ]:
from magic_hour import Client


def generate_image(prompt: str, aspect_ratio: str = "16:9") -> str:
    """Generate an illustration from a text description and save it in output/images.
    Use it only when the user asks for an image, an illustration or a picture.

    Args:
        prompt: what the picture shows and its style, e.g. "a calm illustration of rising bar charts over Jakarta, no text".
        aspect_ratio: "16:9", "1:1" or "9:16".
    """
    os.makedirs("output/images", exist_ok=True)
    client = Client(token=os.getenv("MAGIC_API_KEY"))
    result = client.v1.ai_image_generator.generate(
        image_count=1,
        style={"prompt": prompt},
        aspect_ratio=aspect_ratio,
        download_directory="output/images",
    )
    return f"Image saved: {result.downloaded_paths}"


DESK_PROMPT = SYSTEM_PROMPT + """
Use web search only for news, and name the website of every fact you take from it.
Create images, drafts or e-mails only when the user asks. To e-mail someone, use send_gmail_message;
create a draft only when the user asks for a draft."""

desk_tools = tools + [web_search, generate_image]
desk_agent = Agent(model, tools=desk_tools, toolsets=[gmail], instructions=DESK_PROMPT)

news = await desk_agent.run("What does the web say about Bank OCBC NISP recently? Two sentences, with the websites.")
show_steps(news)

#### Your turn: An illustration for the digest

Ask `desk_agent` for *"an illustration for the top of this morning's bank digest: Indonesian bank stocks, calm and
professional, no text, 16:9"*. Show the steps, then display the newest file in `output/images` with
`IPython.display.Image`. The right result is one `generate_image` call whose `prompt` the agent wrote from your request
(usually longer than yours), and a new image in `output/images`. Each image costs Magic Hour credits: run it once.

In [ ]:
## Your code below



#### Knowledge Check: Two of five Gmail tools

`GmailToolkit` also has tools to search and read your mail. Why does the desk agent get only the draft and send tools?

- **a.** The other tools cost money
- **b.** It needs only those: fewer tools means fewer wrong choices and no access to mail it does not need
- **c.** The model accepts at most two tools from one toolkit
- **d.** The other tools need a different `credentials.json`

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m7_pick", answer)

#### Knowledge Check: Which tool should wait for a person?

Which of the desk agent's tools should **pause for approval** before it runs?

- **a.** `duckduckgo_search`
- **b.** `create_gmail_draft`
- **c.** `send_gmail_message`
- **d.** `get_daily_prices`

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m7_send", answer)

# 7.6 Approval Before Actions

> **The request**
>
> Your manager: **"Reading data is fine. But nothing gets written or sent unless a person says yes."**

> **In this section**
>
> - **Key question:** How do we stop an agent before an action, and let a person decide?
> - **How we run it:** **Live, about 12 minutes.**
> - **What you will do:** pause before `save_note` and `send_gmail_message`, read what the agent wants to do, reject it with a reason,
then review an e-mail and edit it before it goes.
> - **What carries forward:** the approval pattern used in the exercise and in Module 9.

- `Tool(save_note, requires_approval=True)` and `gmail.approval_required(...)` **pause** the run before the tools you
  mark: here the two actions, `save_note` and `send_gmail_message`. Reading data, searching and drafting run without a stop.
- With `output_type=[str, DeferredToolRequests]`, a paused run stops and returns, instead of an answer, a
  `DeferredToolRequests`: its `.approvals` are the actions the agent wants, with their arguments.
- We resume by running again with the message history and our decisions, `DeferredToolResults(approvals={...})`:
  `True` to approve, `ToolDenied("...")` to reject with a message the agent reads, or
  `ToolApproved(override_args={...})` to approve a changed action.

In [ ]:
from pydantic_ai import Tool, DeferredToolRequests, DeferredToolResults, ToolDenied, ToolApproved

approval_tools = [get_company_overview, get_daily_prices, get_top_banks, Tool(save_note, requires_approval=True),
                  web_search, generate_image]
agent = Agent(
    model, tools=approval_tools, instructions=DESK_PROMPT,
    toolsets=[gmail.approval_required(lambda ctx, tool_def, args: tool_def.name == "send_gmail_message")],
    output_type=[str, DeferredToolRequests],
)

paused = await agent.run("Get BBCA's last close, then save it in a note called bbca.txt")

call = paused.output.approvals[0]
print("The agent wants to run:", call.tool_name, call.args_as_dict())

decision = ToolDenied("Do not save yet, I want to check it first.")
result = await agent.run(message_history=paused.all_messages(),
                         deferred_tool_results=DeferredToolResults(approvals={call.tool_call_id: decision}))
print(result.output)
print("File exists:", os.path.exists("output/notes/bbca.txt"))

#### Your turn: Review the e-mail before it goes

In a new conversation (a run with no message history), ask *"E-mail <your MY_EMAIL> BBCA's last close with its date, in two sentences."*
(use `os.getenv("MY_EMAIL")`). The agent gets the price, then pauses before `send_gmail_message`: print the recipient,
subject and message it wants to send. Do not approve it as it is: resume with a `ToolApproved(override_args={...})` decision that keeps every argument but
sets the subject to `IDX Market Assistant: BBCA`.
The right result: the final answer says the e-mail was sent, and it is in your inbox with your subject and BBCA's close.

No Gmail on your laptop? Do the same with `save_note`: ask for a note called `nisp.txt` that says `NISP checked`, and edit
the file name to `nisp_reviewed.txt` before it is saved.

In [ ]:
## Your code below



#### Knowledge Check: Which tools pause?

With `Tool(save_note, requires_approval=True)`, and `approval_required` true only for `send_gmail_message`, which tool
calls wait for a person?

- **a.** All tool calls
- **b.** Only `save_note` and `send_gmail_message`
- **c.** Only the first tool call of each run
- **d.** None: it only logs them

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m7_interrupt", answer)

# 7.7 Memory and Limits

> **The request**
>
> An RM, a minute later: **"And which bank did I just ask about?"**

> **In this section**
>
> - **Key question:** What does an agent remember, and what keeps it safe?
> - **How we run it:** **Live, about 5 minutes.**
> - **What you will do:** ask a follow-up with the message history, and review the limits.
> - **What carries forward:** the safety table you apply to Module 9's scheduled agent.

In [ ]:
result = await agent.run("Which bank did I just ask about?", message_history=result.all_messages())
print(result.output)

`message_history=` gives the agent the earlier messages: that is its **short-term memory**. A run without it starts
empty. The history is only a Python variable: it lasts only while the program runs.

| Risk | Control |
|---|---|
| A made-up number | Rule in the system prompt; check against the `ToolReturnPart`s (7.3) |
| An action nobody wanted | Approval for action tools (7.6); read and action tools kept apart |
| An e-mail to the wrong person | Approval on send; a person reads the recipient and the text; drafts need no approval |
| Facts from the web | Web search for news only, with the website named; numbers from Sectors |
| Too many steps, too much cost | Few tools with clear descriptions; `usage_limits=UsageLimits(request_limit=...)` on the run |
| Advice | Forbidden in the system prompt; read the text before it is sent |

#### Knowledge Check: A restart

The agent is paused for approval and you restart the kernel before answering. What happens to the paused run?

- **a.** It continues by itself
- **b.** The note is saved anyway
- **c.** It is lost: the message history was only a variable in memory
- **d.** It is written to `.env`

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m7_restart", answer)

#### Knowledge Check: Scheduler or agent?

What is the difference between the Module 3 scheduler and the agent?

- **a.** The scheduler decides WHEN fixed code runs; the agent decides WHAT data to get next
- **b.** None
- **c.** The agent decides WHEN
- **d.** The scheduler uses an LLM

Type your choice (`a`, `b`, `c` or `d`) between the quotes, then run the cell.

In [ ]:
## Your answer: a, b, c or d
answer = ""
check_answer("m7_when_what", answer)

# Module 7 Wrap-up

| Idea | In code |
|---|---|
| The loop | `Agent(model, tools=..., instructions=...)`; read it with `result.all_messages()` |
| The rules | the system prompt; check numbers against `ToolReturnPart`s |
| Ready-made tools | `duckduckgo_search_tool()`, `GmailToolkit().get_tools()` in `LangChainToolset`; any API wrapped in a plain function |
| Fields for code | `output_type=` → `result.output` |
| Approval | `requires_approval=True` + `output_type=[str, DeferredToolRequests]`; `DeferredToolResults(approvals={...})` |
| Memory | `message_history=result.all_messages()` |

**Next:** `day2/examples/m7_example.py` (approve one save, reject one), then exercise part 2. In Module 8 the agent gets
tools from Sectors' MCP server instead of our own functions.